In [2]:
import pandas as pd

In [9]:
# manually adding the names and ids
path = '/Users/valeriia/Downloads/Padel_results_2.xlsx'
xls = pd.ExcelFile(path)
df_rating = pd.read_excel(xls, 'Rating').set_index('ID')
df_rating = df_rating.sort_index(ascending=True)



df_rating.head()


,Name,Rating,ELO_coef_K,Rating_change
ID,,,,
1,Andrej,2405,40,NaN
2,Armen,2350,40,20.0
3,Vladlen,2367,40,-24.0
5,Sasha,2307,40,NaN
7,Marton,2646,40,NaN


In [2]:
def calculate_elo_rating(rating_1, rating_2, score):
    # Calculate the expected scores
    E_1 = 1 / (1 + 10 ** ((rating_2 - rating_1) / 400))
    E_2 = 1 / (1 + 10 ** ((rating_1 - rating_2) / 400))
    
    # Calculate the ratings' change
    rating_change_1 = score - E_1
    rating_change_2 = (1 - score) - E_2
    
    return round(rating_change_1, 1), round(rating_change_2, 1)

In [3]:
# path = 'C:/Users/Vladb/Downloads/Padel_results_2.xlsx'
# xls = pd.ExcelFile(path)
# df_rating = pd.read_excel(xls, 'Rating').set_index('ID')
# df_results = pd.read_excel(xls, 'Results')[['Pair_1_player_1', 'Pair_1_player_2', 'Pair_2_player_1',
#        'Pair_2_player_2', 'Result']].dropna()

In [11]:
rating_dict = df_rating.Rating.to_dict()
rating_change_dict = {key: 0 for key, value in rating_dict.items()}

In [7]:
for i in range(len(df_results)):
    player_1 = df_results.iloc[i]['Pair_1_player_1']
    player_2 = df_results.iloc[i]['Pair_1_player_2']
    player_3 = df_results.iloc[i]['Pair_2_player_1']
    player_4 = df_results.iloc[i]['Pair_2_player_2']
    
    pair_1 = (rating_dict[player_1] + rating_dict[player_2])/2
    pair_2 = (rating_dict[player_3] + rating_dict[player_4])/2
    
    score = df_results.iloc[i]['Result']
    
    rating_change_1, rating_change_2 = calculate_elo_rating(pair_1, pair_2, score)
    
    rating_change_dict[player_1] += rating_change_1
    rating_change_dict[player_2] += rating_change_1
    rating_change_dict[player_3] += rating_change_2
    rating_change_dict[player_4] += rating_change_2

In [8]:
rating_change_dict = {key: round(float(value), 1) for key, value in rating_change_dict.items()}

In [9]:
df_rating_change = pd.DataFrame.from_dict(rating_change_dict, orient='index', columns=['Rating']
                                          ).replace(0.0, '')

In [10]:
df_rating['Rating_change'] = df_rating_change.Rating * df_rating.ELO_coef_K

In [11]:
df_rating.head(10)

,Name,Rating,ELO_coef_K,Rating_change
ID,,,,
7,Marton,2646,40,
1,Andrej,2405,40,
167,Mauricio,2378,40,
3,Vladlen,2367,40,-24.0
153,Nikola,2357,40,
2,Armen,2350,40,20.0
237,Laura,2340,40,
5,Sasha,2307,40,
206,Fernando,2300,40,


In [12]:
with pd.ExcelWriter(path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_rating.to_excel(writer, sheet_name='Rating', index_label='ID')